# Posterior error: compression plus conditional model fitting

[Formal argument](../09_conditioning_information_loss.md). Enumerated finite joint distributions test the equality; no HSE or diffusion model is trained.

In [ ]:
import numpy as np

def binary_kl(p, q):
    return p * np.log(p / q) + (1-p) * np.log((1-p)/(1-q))

p_full = np.array([.9, .1])
p_obs = np.array([.5, .5])
p_compressed, q_fit = .5, .7
compression = float(p_obs @ binary_kl(p_full, p_compressed))
fitting = float(binary_kl(p_compressed, q_fit))
total = float(p_obs @ binary_kl(p_full, q_fit))
np.testing.assert_allclose(total, compression + fitting, rtol=0, atol=1e-12)
print({'total_nats': total, 'compression_nats': compression, 'fitting_nats': fitting,
       'identity_residual': abs(total-compression-fitting)})

## Side information changes the condition, not the target
Keep the same raw observations. Let the decoder side information now expose their binary value. Compression is zero, but an unchanged fitted predictor may remain wrong.

In [ ]:
p_with_side_information = p_full.copy()
compression_with_a = float(p_obs @ binary_kl(p_full, p_with_side_information))
fitting_with_a = float(p_obs @ binary_kl(p_with_side_information, q_fit))
np.testing.assert_allclose(compression_with_a, 0., atol=1e-12)
np.testing.assert_allclose(fitting_with_a, total, atol=1e-12)
print('Full side information: compression =', compression_with_a, '; fitting =', fitting_with_a)

## Nonconstant, noninjective H
Four observations form two token classes. The compressed conditional must be computed from the joint distribution; substituting an arbitrary plug-in posterior would not verify this theorem.

In [ ]:
p_obs = np.array([.2, .3, .1, .4])
p_full = np.array([.9, .1, .7, .3])
h = np.array([0, 0, 1, 1])
q_by_h = np.array([.6, .4])
p_by_h = np.array([(p_obs[h == k] @ p_full[h == k]) / p_obs[h == k].sum() for k in [0, 1]])
compression2 = float(p_obs @ binary_kl(p_full, p_by_h[h]))
fitting2 = float(p_obs @ binary_kl(p_by_h[h], q_by_h[h]))
total2 = float(p_obs @ binary_kl(p_full, q_by_h[h]))
np.testing.assert_allclose(total2, compression2 + fitting2, rtol=0, atol=1e-12)
assert compression2 > 0 and fitting2 > 0
print({'P(Z0=1|H)': p_by_h.tolist(), 'total_nats': total2,
       'compression_nats': compression2, 'fitting_nats': fitting2})

**Conclusion.** A richer decoder cannot remove the fixed condition’s information loss. The generic identity is prior analytical machinery, not the proposed method novelty. Admission still needs an actual same-budget conditioner gain.

In [ ]:
print("THEORY_DEMO_PASS::09_conditioning_information_loss")